In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS gold
""")

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 3, Finished, Available, Finished, False)

DataFrame[]

### dim_customer table

In [2]:
gold_dim_customer = spark.table("silver.slv_customer")
gold_dim_customer.show(10)


StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 4, Finished, Available, Finished, False)

+-----------+--------------------+-------------+---------------+-------------+---------+--------------------+--------------------+
|CUSTOMER_ID|       CUSTOMER_NAME|REGISTERED_ON|KYC_VERIFIED_ON|CUSTOMER_TYPE|     CITY|         ingested_at|         source_file|
+-----------+--------------------+-------------+---------------+-------------+---------+--------------------+--------------------+
|     C20000|          Diya Mehta|   2024-10-09|     2024-11-22|   Individual| Lonavala|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20001|      Sterling Infra|   2023-06-21|     2023-07-06|      Company| Baramati|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20002| Trident Civil Works|   2023-06-14|     2023-07-13|      Company|Chinchwad|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20003|        Rahul Mehta |   2024-10-10|     2024-10-16|   Contractor|  Kharadi|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20004|       Vivaan Pillai|   2025-01-14|     2025-02-19|   Contractor| Lona

In [3]:
from pyspark.sql import functions as F

# Calculate the customer's tenure in completed years
gold_dim_customer = gold_dim_customer.withColumn(
    "tenure_years",
    F.floor(F.months_between(F.current_date(), F.col("REGISTERED_ON")) / 12)
)

# Group customers into tenure bands
gold_dim_customer = gold_dim_customer.withColumn(
    "tenure_band",
    F.when(F.col("tenure_years").isNull(), "Unknown")
     .when(F.col("tenure_years") < 1, "Less than 1 Year")
     .when(F.col("tenure_years") < 3, "1-2 Years")
     .when(F.col("tenure_years") < 5, "3-4 Years")
     .otherwise("5+ Years")
)

display(gold_dim_customer)

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 050c8d1d-4df1-4080-a39e-99d1c7c375a7)

In [4]:
gold_dim_customer.write\
.format('delta')\
.mode('overwrite')\
.option('mergeSchema','true')\
.saveAsTable('gold.gold_dim_customer')

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 6, Finished, Available, Finished, False)

### dim_depot table

In [5]:
gold_dim_depot = spark.table('silver.slv_depot')
gold_dim_depot.show()


StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 7, Finished, Available, Finished, False)

+----------+--------------+-----+----------+--------------------+--------------------+
|DEPOT_CODE|    DEPOT_NAME| ZONE|FLEET_SIZE|         ingested_at|         source_file|
+----------+--------------+-----+----------+--------------------+--------------------+
|      HADP|Hadapsar Depot| East|        55|2026-10-13 10:47:...|abfss://c9364e95-...|
|      CHKN|  Chakan Depot|North|        48|2026-10-13 10:47:...|abfss://c9364e95-...|
|      WAKD|   Wakad Depot| West|        42|2026-10-13 10:47:...|abfss://c9364e95-...|
|      KHRD| Kharadi Depot| East|        30|2026-10-13 10:47:...|abfss://c9364e95-...|
|      TLGN|Talegaon Depot|North|        25|2026-10-13 10:47:...|abfss://c9364e95-...|
|      BRMT|Baramati Depot|South|        20|2026-10-13 10:47:...|abfss://c9364e95-...|
+----------+--------------+-----+----------+--------------------+--------------------+



In [6]:
gold_dim_depot.write\
.format('delta')\
.mode('overwrite')\
.option('overwriteSchema','true')\
.saveAsTable('gold.gold_dim_depot')

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 8, Finished, Available, Finished, False)

### dim_date table

In [7]:
from pyspark.sql import functions as F

# Generate dates for the last 30 days, including today

gold_dim_date = spark.range(1).select(
    F.explode(
        F.sequence(
            F.date_sub(F.current_date(), 30),
            F.current_date(),
            F.expr("INTERVAL 1 DAY")
        )
    ).alias("calendar_date")
)

# Add useful calendar attributes
gold_dim_date = (
    gold_dim_date
    .withColumn("date_key", F.date_format("calendar_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("calendar_date"))
    .withColumn("month", F.month("calendar_date"))
    .withColumn("day_of_month", F.dayofmonth("calendar_date"))
    .withColumn("day_of_week", F.dayofweek("calendar_date"))
    .withColumn("month_name", F.date_format("calendar_date", "MMMM"))
    .withColumn("quarter", F.quarter("calendar_date"))
    .withColumn("week_of_year", F.weekofyear("calendar_date"))
    .withColumn("is_weekend", F.dayofweek("calendar_date").isin([1, 7]))
)

# Display the generated calendar
gold_dim_date.show(31, truncate=False)

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 9, Finished, Available, Finished, False)

+-------------+--------+----+-----+------------+-----------+----------+-------+------------+----------+
|calendar_date|date_key|year|month|day_of_month|day_of_week|month_name|quarter|week_of_year|is_weekend|
+-------------+--------+----+-----+------------+-----------+----------+-------+------------+----------+
|2026-09-17   |20260917|2026|9    |17          |5          |September |3      |38          |false     |
|2026-09-18   |20260918|2026|9    |18          |6          |September |3      |38          |false     |
|2026-09-19   |20260919|2026|9    |19          |7          |September |3      |38          |true      |
|2026-09-20   |20260920|2026|9    |20          |1          |September |3      |38          |true      |
|2026-09-21   |20260921|2026|9    |21          |2          |September |3      |39          |false     |
|2026-09-22   |20260922|2026|9    |22          |3          |September |3      |39          |false     |
|2026-09-23   |20260923|2026|9    |23          |4          |Sept

In [8]:
gold_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.gold_dim_date")

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 10, Finished, Available, Finished, False)

### fact_rental table

In [9]:
gold_fact_rental = spark.table('silver.slv_rental')
display(gold_fact_rental)

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8b8e0600-7c04-4489-80e4-1059ff8fad53)

In [10]:
from pyspark.sql import functions as F

# Load all rentals from the Silver table
gold_fact_rental = spark.table('silver.slv_rental')
gold_dim_date = spark.table('gold.gold_dim_date')

# Get the start and end dates of the 30-day reporting window
window_dates = gold_dim_date.agg(
    F.min('calendar_date').alias('window_start'),
    F.max('calendar_date').alias('window_end')
)

# Add reporting window dates to every rental
gold_fact_rental = gold_fact_rental.crossJoin(window_dates)

# Add the three required columns
gold_fact_rental = (
    gold_fact_rental

    # Calculate rental duration; keep NULL if the machine is still out
    .withColumn(
        'rental_duration_days',
        F.when(
            F.col('checkin_ts').isNotNull(),
            F.datediff(
                F.to_date('checkin_ts'),
                F.to_date('checkout_ts')
            )
        )
    )

    # True means returned; False means still out
    .withColumn(
        'is_returned',
        F.col('checkin_ts').isNotNull()
    )

    # Count only rental days inside the reporting window
    .withColumn(
        'asset_days_in_window',
        F.greatest(
            F.lit(0),
            F.datediff(
                F.least(
                    F.coalesce(
                        F.to_date('checkin_ts'),
                        F.date_add('window_end', 1)
                    ),
                    F.date_add('window_end', 1)
                ),
                F.greatest(
                    F.to_date('checkout_ts'),
                    F.col('window_start')
                )
            )
        )
    )

    # Remove temporary reporting window columns
    .drop('window_start', 'window_end')
)

# Check the new columns
display(gold_fact_rental)

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, da8e4242-d35d-4801-bc66-eb896005edb8)

In [12]:
gold_fact_rental.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('gold.gold_fact_rental')

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 15, Finished, Available, Finished, False)

### fact_billing table

In [13]:
# Read parsed billing data from Silver
gold_fact_billing = spark.table('silver.slv_billing')

# Save as a Gold Delta table
gold_fact_billing.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable('gold.gold_fact_billing')

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 16, Finished, Available, Finished, False)

In [14]:
display(gold_fact_billing)

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5c401953-02e1-44b9-ae23-0f1b16ecb05a)

In [15]:
# Optimize rental fact table
spark.sql("""
    OPTIMIZE gold.gold_fact_rental
    ZORDER BY (rental_id)
""")

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 18, Finished, Available, Finished, False)

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint,incrementalClusteringStats:struct<autoReclusterEnabled:boolean,overlapThreshold:double,partialZCubeFound:boolean,partialZCubeSize:bigint,totalFilesInTable:bigint,filesPassedToZCubeFilter:bigint,filesSkippedMissingStats:int,autoReclusterFilesAdded:int,a

In [16]:
# Optimize billing fact table
spark.sql("""
    OPTIMIZE gold.gold_fact_billing
    ZORDER BY (RENTAL_ID)
""")

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 19, Finished, Available, Finished, False)

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint,incrementalClusteringStats:struct<autoReclusterEnabled:boolean,overlapThreshold:double,partialZCubeFound:boolean,partialZCubeSize:bigint,totalFilesInTable:bigint,filesPassedToZCubeFilter:bigint,filesSkippedMissingStats:int,autoReclusterFilesAdded:int,a

In [17]:
display(spark.sql("""
    DESCRIBE HISTORY gold.gold_fact_rental
"""))

display(spark.sql("""
    DESCRIBE HISTORY gold.gold_fact_billing
"""))

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c166b406-e375-40e5-91f9-779834e0b993)

SynapseWidget(Synapse.DataFrame, bc4aca4a-f23e-4e89-834b-5e7ed82ab0ff)

In [18]:
spark.sql("SHOW TABLES IN gold").show(truncate=False)

spark.table("gold.gold_fact_rental").printSchema()
spark.table("gold.gold_fact_billing").printSchema()

StatementMeta(, 51c85a8b-68cb-4810-b14e-4440201537f3, 21, Finished, Available, Finished, False)

+------------------------------+-----------------+-----------+
|namespace                     |tableName        |isTemporary|
+------------------------------+-----------------+-----------+
|ws_buildmate.lh_buildmate.gold|gold_dim_customer|false      |
|ws_buildmate.lh_buildmate.gold|gold_dim_date    |false      |
|ws_buildmate.lh_buildmate.gold|gold_dim_depot   |false      |
|ws_buildmate.lh_buildmate.gold|gold_fact_billing|false      |
|ws_buildmate.lh_buildmate.gold|gold_fact_rental |false      |
+------------------------------+-----------------+-----------+

root
 |-- rental_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- depot_code: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- checkout_ts: timestamp (nullable = true)
 |-- checkin_ts: timestamp (nullable = true)
 |-- rental_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- duplicate_flag: string (nullable 